In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import h5py

In [ ]:
def inspect_hdf5(path_h5:str):
    """Recursively print all fields in the hdf5 with their shape and dtype"""
    def inspect_hdf5_rec(item):
        if isinstance(item, h5py.Dataset):
            print(item.name, item.shape, item.dtype, item.chunks)
        else:
            for child in item.values():
                inspect_hdf5_rec(child)
    with h5py.File(path_h5, "r") as file_h5:
        inspect_hdf5_rec(file_h5)
    print()

In [ ]:
prefix_refHDF5 = "../Data/reference/1000g_maf5_hdf5/maf5_chr"

sample_name = "SB606"
path_bam = f"../Data/ExampleData/med_jews/bam/{sample_name}.merged.rg.markdup.indrealn_recalibrated.bam"
dir_sampleHDF5 = f"../Data/ExampleData/med_jews/HDF5"

In [ ]:
inspect_hdf5(prefix_refHDF5+"1.hdf5")

# Bam to hdf5

In [ ]:
from hapROH.utils.bam2hdf5 import bam2hdf5s

In [ ]:
bam2hdf5s(path_bam, prefix_refHDF5, dir_sampleHDF5, sample_name, overwrite=False)

In [ ]:
chrom = 1
path_sample = os.path.join(dir_sampleHDF5, f"{sample_name}.chr{chrom}.hdf5")
path_ref = prefix_refHDF5 + f"{chrom}.hdf5"

inspect_hdf5(path_sample)
inspect_hdf5(path_ref)

## Time loading data from hdf5

In [ ]:
chrom = 1
path_ref = prefix_refHDF5 + f"{chrom}.hdf5"

file_ref = h5py.File(path_ref)

In [ ]:
file_ref["calldata/GT"].shape

In [ ]:
nb_subset = 15580
nb_total = file_ref["calldata/GT"].shape[0]
idx_samples = np.random.choice(nb_total, nb_subset,replace=False)
idx_samples.sort()
idx_samples

In [ ]:
%%time
# load first all samples, then subset
data = file_ref["calldata/GT"][:]
datat = data[idx_samples]

In [ ]:
%%time
# load only subset: slightly slower
data = file_ref["calldata/GT"][idx_samples,:,:]

# Call ROH
Run on eigenstrat from the levant

In [ ]:
chrom = 19
sample_name = "I1178"
path_sample = "../Data/ExampleData/Levant_ChL/Levant_ChL"   # The path before the .ind, .snp, .geno
path_ref = "../Data/reference/1000g_maf5_hdf5/maf5_chr"    # The path up to the chr. number
path_meta_ref = "../Data/reference/meta_1000g.csv"

folder_base = "../Data/Output/Levant"  # Folder where you want to save the results to 

## Current method

In [ ]:
from hapsburg.PackagesSupport.hapsburg_run import hapsb_ind

In [ ]:
%%time
folder_out_roh = folder_base + "_old_version"
df_roh = hapsb_ind(iid=sample_name, chs=[chrom], 
           path_targets=path_sample,
           h5_path1000g=path_ref,
           meta_path_ref=path_meta_ref, 
           folder_out=folder_out_roh,
           processes=1, output=True,
           readcounts=False, logfile=True, combine=True)

## Under development

In [ ]:
from hapROH.run_new import callROH_chr

In [ ]:
%%time
folder_out_roh = folder_base + "_test"
post_pb = callROH_chr(path_sample, path_ref+f"{chrom}.hdf5", chrom, sample_name,
            folder_out=folder_out_roh + "_test", logfile=None, loglevel=2)

## Plot results

In [ ]:
from hapsburg.figures.plot_posterior import plot_posterior_cm

In [ ]:
folder_plot = os.path.join(folder_out_roh, sample_name, "chr" + str(chrom), "")
plot_posterior_cm(folder = folder_plot, savepath="", 
                  empirical=True, m=1, cm_lim=[], groundtruth = False, min_cm=1,
                  readcount=False, figsize=(10,4), title=f"{sample_name}, Chromosome {chrom}")